# TP14 — Prédire le Nutri-Score

**Objectif :** comparer des classifieurs A–E sans fuite de données, puis expliquer les erreurs observées.

Ce notebook prépare le protocole et les contrôles. Les cellules **À FAIRE** te laissent les entraînements, choix, mesures et analyses qui constituent le cœur du TP. Consigne les résultats dans le tableau comparatif associé.

Données : Open Food Facts contributors — licence ODbL 1.0.

## Protocole

- Conserver le split fourni par le TP13 ; ne pas le recalculer.
- Ajuster imputation, standardisation, encodage et modèle dans chaque pli de train.
- Exclure nutriscore_score, code, product_name, brands, split et la cible des features.
- Réutiliser les mêmes plis stratifiés et les mêmes métriques pour chaque modèle.
- Garder le test à l’écart jusqu’au choix du modèle ; l’évaluer une seule fois.
- Écrire les résultats avant leur interprétation.

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, make_scorer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file()), Path.cwd())
print(f'Racine : {ROOT.resolve()}')

## 1. Charger et contrôler les données

Le dépôt peut contenir l’échantillon local ou un export daté du TP13. Si nécessaire, définis NUTRISCOPE_DATA avec le chemin du Parquet.

In [ ]:
candidats = [
    Path(os.environ['NUTRISCOPE_DATA']).expanduser() if os.environ.get('NUTRISCOPE_DATA') else None,
    ROOT / 'data' / 'nutriscope_ml_sample.parquet',
    ROOT / 'data' / 'tp13' / 'exports' / 'features_nutriscore_20261125.parquet',
    ROOT / 'data-nutriscope' / 'ml' / 'nutriscope_ml_sample.parquet',
]
DATA_PATH = next((p for p in candidats if p is not None and p.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Parquet ML introuvable. Définis NUTRISCOPE_DATA.')
df = pd.read_parquet(DATA_PATH)
print(f'{DATA_PATH.resolve()} — {len(df):,} lignes × {len(df.columns)} colonnes')
display(df.head())

In [ ]:
TARGET, SPLIT = 'nutriscore_grade', 'split'
NEVER_FEATURES = {'nutriscore_score', 'code', 'product_name', 'brands', SPLIT, TARGET}
NUMERIC_FEATURES = [
    'energy-kcal_100g', 'fat_100g', 'saturated-fat_100g', 'carbohydrates_100g',
    'sugars_100g', 'proteins_100g', 'salt_100g', 'fiber_100g',
    'fruits-vegetables-legumes_100g', 'nova_group', 'additives_n',
]
CATEGORICAL_FEATURES = ['pnns_groups_1', 'pnns_groups_2']
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
missing = {TARGET, SPLIT, *FEATURES}.difference(df.columns)
assert not missing, f'Colonnes absentes : {sorted(missing)}'
assert set(df[TARGET].dropna().unique()) <= set('abcde'), 'Grade inattendu'
assert set(df[SPLIT].dropna().unique()) == {'train', 'test'}, 'Split train/test attendu'
display(df.groupby([SPLIT, TARGET], observed=True).size().unstack(fill_value=0))
display(df[FEATURES].isna().mean().sort_values(ascending=False).rename('part_manquante'))

### Lecture des données — À FAIRE

Note le volume réellement lu, la distribution des grades et les variables les plus incomplètes. Vérifie que le fichier correspond au split du TP13 ; les nombres de l’énoncé sont des repères, pas des résultats à recopier.

In [ ]:
train_df = df.loc[df[SPLIT].eq('train')].copy()
# Le test reste dans df et n’est pas transformé avant le choix final.
X_train = train_df[FEATURES].copy()
y_train = train_df[TARGET].copy()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
folds = list(cv.split(X_train, y_train))  # mêmes indices pour chaque modèle
SCORING = {
    'f1_macro': make_scorer(f1_score, average='macro'),
    'accuracy': make_scorer(accuracy_score),
    'balanced_accuracy': make_scorer(balanced_accuracy_score),
}
print(f'Train : {len(X_train):,} lignes ; plis CV : {len(folds)}')

## 2. Baselines — À FAIRE

Évalue en validation croisée la classe majoritaire, le Dummy stratifié et des arbres de décision de profondeurs 3 et 8. Utilise cross_validate avec X_train, y_train, cv=folds et SCORING. Consigne moyenne et écart-type du F1 macro, accuracy, balanced accuracy et temps dans le tableau.

**À commenter :** pourquoi l’accuracy de la baseline majoritaire paraît-elle correcte alors que le F1 macro est faible ? L’arbre peu profond prédit-il toutes les classes ?

In [ ]:
# À FAIRE : construire et comparer les quatre estimateurs de baseline.
# Conserver fit_time et score_time retournés par cross_validate.
raise NotImplementedError('Entraîner et consigner les baselines TP14')

## 3. Pipeline et modèles — À FAIRE

Construis un ColumnTransformer puis un Pipeline complet pour chaque modèle : imputation des variables optionnelles avec indicateur de manque, standardisation numérique, one-hot des catégories avec regroupement des modalités rares, puis classifieur. Les ratios ou écarts énergétiques doivent être créés dans le pipeline avec FunctionTransformer. Entraîne la régression logistique et la forêt aléatoire sur les mêmes plis.

Écris le contrôle qui prouve que le préprocesseur ajusté sur train n’a pas appris les statistiques du test.

In [ ]:
# À FAIRE : construire la préparation et les deux pipelines complets.
# Évaluer avec les plis et métriques déjà définis.
raise NotImplementedError('Construire, évaluer et contrôler les pipelines')

## 4. Comparaison et déséquilibre — À FAIRE

Rassemble tous les résultats dans un seul tableau. Ajoute la matrice de confusion du meilleur modèle CV, normalisée par ligne. Mesure le rapport majoritaire/minoritaire et compare class_weight='balanced'. Si tu essaies SMOTE, place-le à l’intérieur du pipeline.

Explique l’écart entre accuracy et F1 macro ; indique si le traitement du déséquilibre apporte un gain pertinent au regard de la variabilité entre plis.

## 5. Fuite et importances — À FAIRE

Reproduis volontairement la fuite en ajoutant nutriscore_score. Mesure son F1 macro et son importance par permutation sur une validation interne au train. Explique pourquoi ce score est à l’origine du grade. Retire-le et écris un test anti-fuite, avec un contrôle négatif sur une feature légitime.

Sur le modèle honnête, calcule les importances par permutation hors entraînement et confronte-les aux composantes officielles : énergie, sucres, acides gras saturés, sel, fibres, protéines et fruits/légumes/légumineuses. Distingue fuite et proxy métier.

In [ ]:
# À FAIRE : fuite volontaire, importances sur validation interne, contrôles et interprétation.
raise NotImplementedError('Mener l’analyse de fuite et d’importance')

## 6. Test final et trois erreurs — À FAIRE

Verrouille les choix sur la CV, ajuste le modèle retenu sur tout le train, puis construis X_test et consulte le test une seule fois. Inscris les métriques avant leur interprétation.

Choisis trois erreurs réelles de natures différentes. Pour chaque produit : nutriments, explication de la prédiction, diagnostic (apprentissage, fiche erronée, règle absente des features ou référentiel différent) et action. Au moins un diagnostic doit dépasser « le modèle doit mieux apprendre ».

In [ ]:
# À FAIRE : dernier fit sur tout le train puis évaluation unique sur test.
# Conserver code, nom et nutriments pour analyser les erreurs à la main.
raise NotImplementedError('Évaluer le modèle retenu et analyser trois erreurs')

## Conclusion — À FAIRE

Résume le meilleur modèle et ses chiffres CV/test, la variabilité, le piège de l’accuracy, la fuite détectée/corrigée et les trois diagnostics d’erreurs. Vérifie l’écart CV/test selon l’énoncé. Après les mesures réelles, reporte la phrase de synthèse dans docs/journal.md.